# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%reload_ext dotenv
%dotenv ../05_src/.secrets

In [2]:
import os
api_key = os.getenv("API_GATEWAY_KEY")
print(f"Key found: {api_key is not None}")
print(f"Key starts with: {api_key[:15] if api_key else 'NO KEY FOUND'}...")

Key found: True
Key starts with: O6SxAEk8kmqIs2I...


In [3]:
# ============================================
# ALL IMPORTS
# ============================================

import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional
import tiktoken

# LangChain imports for document loading
from langchain_community.document_loaders import WebBaseLoader

# DeepEval imports for evaluation
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase

print("All imports successful!")

# ============================================
# INITIALIZE OPENAI CLIENT
# ============================================
import os
from openai import OpenAI

# THIS IS THE CORRECT SETUP FOR THE COURSE GATEWAY
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
    api_key=api_key,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

MODEL_NAME = "gpt-4o-mini"

print("Client configured correctly!")

USER_AGENT environment variable not set, consider setting it to identify your requests.


All imports successful!
Client configured correctly!


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [4]:
# ============================================
# PYDANTIC MODEL FOR STRUCTURED OUTPUT
# ============================================

class ArticleSummary(BaseModel):
    """Structured output model for article summaries."""
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(description="Why this article is relevant for an AI professional")
    Summary: str = Field(description="Concise summary, no longer than 1000 tokens")
    Tone: str = Field(description="The tone used to produce the summary")
    InputTokens: int = Field(description="Number of input tokens used")
    OutputTokens: int = Field(description="Number of tokens in the output")

print("Pydantic model defined!")

Pydantic model defined!


In [5]:
url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"
loader = WebBaseLoader(url)
docs = loader.load()

document_text = ""
for doc in docs:
    document_text += doc.page_content + "\n"

print(f"Document loaded! Length: {len(document_text)} characters")

Document loaded! Length: 35720 characters


In [6]:
def generate_summary(document_text: str, tone_description: str) -> dict:
    
    developer_prompt = """You are an expert AI assistant specializing in document analysis and summarization. 
Your task is to analyze the provided document and produce a structured output.

IMPORTANT GUIDELINES:
1. Extract or infer the author and title from the document.
2. Write a one-paragraph explanation of why this article is relevant for an AI professional.
3. Write a concise summary (under 1000 tokens).
4. Use the EXACT tone specified by the user.
5. Be accurate and do not hallucinate information."""

    user_prompt = f"""Please analyze the following document and provide a structured response.

DOCUMENT TEXT:
{document_text[:15000]}

TONE REQUIREMENT: Write the summary using the following tone: {tone_description}

Please ensure the summary is accurate and properly reflects the document content."""

    completion = client.beta.chat.completions.parse(
        model=MODEL_NAME,
        messages=[
            {"role": "developer", "content": developer_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format=ArticleSummary,
        temperature=0.3
    )
    
    result = completion.choices[0].message.parsed
    
    output_dict = {
        "Author": result.Author,
        "Title": result.Title,
        "Relevance": result.Relevance,
        "Summary": result.Summary,
        "Tone": result.Tone,
        "InputTokens": completion.usage.prompt_tokens,
        "OutputTokens": completion.usage.completion_tokens
    }
    
    return output_dict

print("Generation function defined!")

Generation function defined!


In [7]:
tone_choice = "Victorian English - use elaborate, ornate language characteristic of 19th century British literature"

print("Generating summary... please wait...")
summary_result = generate_summary(document_text, tone_choice)

print("\n" + "=" * 80)
print("GENERATED SUMMARY RESULTS")
print("=" * 80)
print(f"Author: {summary_result['Author']}")
print(f"Title: {summary_result['Title']}")
print(f"\nRelevance:\n{summary_result['Relevance']}")
print(f"\nTone Used: {summary_result['Tone']}")
print(f"\nSummary:\n{summary_result['Summary']}")
print(f"\nInput Tokens: {summary_result['InputTokens']}")
print(f"Output Tokens: {summary_result['OutputTokens']}")

Generating summary... please wait...

GENERATED SUMMARY RESULTS
Author: Alex Ross
Title: What Is Noise?

Relevance:
This article is of profound significance to an AI professional as it delves into the multifaceted concept of noise, not merely in the auditory sense but as a pervasive condition that affects data and information processing. Understanding the implications of noise in both sound and data is crucial for developing algorithms that can discern meaningful signals amidst the cacophony of irrelevant information, thereby enhancing the efficacy of AI systems in various applications.

Tone Used: Victorian English

Summary:
In the grand tapestry of existence, the term 'noise' emerges as a most perplexing and multifarious concept, oscillating between the realms of delightful melody and maddening cacophony. The esteemed author, Alex Ross, embarks upon a profound exploration of this auditory phenomenon, tracing its etymological roots to notions of nuisance and nausea, whilst simultaneou

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [10]:
# ============================================
# EVALUATION
# ============================================

from deepeval.models import GPTModel
from deepeval.test_case import LLMTestCaseParams

# Setup evaluation model with course gateway
eval_model = GPTModel(
    model="gpt-4o-mini",
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
)

test_case = LLMTestCase(
    input=document_text[:5000],
    actual_output=summary_result["Summary"]
)

# Summarization Metric
summarization_metric = SummarizationMetric(
    threshold=0.5,
    assessment_questions=[
        "Does the summary capture the main arguments?",
        "Is the summary concise?",
        "Does it avoid new information?",
        "Is the structure logical?",
        "Does it represent the original scope?"
    ],
    model=eval_model,
    verbose_mode=False
)
summarization_metric.measure(test_case)
print(f"SUMMARIZATION SCORE: {summarization_metric.score:.3f}")

# Coherence Metric
coherence_metric = GEval(
    name="Coherence",
    criteria="Does the summary flow logically and connect ideas well?",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model
)
coherence_metric.measure(test_case)
print(f"COHERENCE SCORE: {coherence_metric.score:.3f}")

# Tonality Metric
tonality_metric = GEval(
    name="Tonality",
    criteria="Does the summary maintain Victorian English tone consistently?",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model
)
tonality_metric.measure(test_case)
print(f"TONALITY SCORE: {tonality_metric.score:.3f}")

# Safety Metric
safety_metric = GEval(
    name="Safety",
    criteria="Is the summary free of harmful or inappropriate content?",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=eval_model
)
safety_metric.measure(test_case)
print(f"SAFETY SCORE: {safety_metric.score:.3f}")

Output()

Output()

SUMMARIZATION SCORE: 0.438


Output()

COHERENCE SCORE: 0.682


Output()

TONALITY SCORE: 0.853


SAFETY SCORE: 0.876


In [12]:
# ============================================
# ENHANCEMENT - Self-correction
# ============================================

print("Generating enhanced summary...")

# Store original scores first
orig_summarization = summarization_metric.score
orig_coherence = coherence_metric.score
orig_tonality = tonality_metric.score
orig_safety = safety_metric.score

# Create improved summary
improve_prompt = f"""Please improve this summary based on the evaluation feedback below. Maintain Victorian English tone.

EVALUATION FEEDBACK:
- Summarization score was {orig_summarization:.3f}. Improve accuracy and coverage.
- Coherence score was {orig_coherence:.3f}. Improve flow and connections between ideas.

ORIGINAL SUMMARY:
{summary_result['Summary']}

Provide an improved version addressing the feedback. Keep Victorian English tone."""

completion = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "developer", "content": "You improve summaries based on feedback."},
        {"role": "user", "content": improve_prompt}
    ],
    temperature=0.3
)

enhanced_summary = completion.choices[0].message.content

# Evaluate enhanced summary
enhanced_test_case = LLMTestCase(
    input=document_text[:5000],
    actual_output=enhanced_summary
)

summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)

# Compare
print("\n" + "=" * 60)
print("COMPARISON: ORIGINAL vs ENHANCED")
print("=" * 60)
print(f"{'Metric':<20} {'Original':<10} {'Enhanced':<10} {'Change':<10}")
print("-" * 50)
print(f"{'Summarization':<20} {orig_summarization:<10.3f} {summarization_metric.score:<10.3f} {summarization_metric.score - orig_summarization:+.3f}")
print(f"{'Coherence':<20} {orig_coherence:<10.3f} {coherence_metric.score:<10.3f} {coherence_metric.score - orig_coherence:+.3f}")
print(f"{'Tonality':<20} {orig_tonality:<10.3f} {tonality_metric.score:<10.3f} {tonality_metric.score - orig_tonality:+.3f}")
print(f"{'Safety':<20} {orig_safety:<10.3f} {safety_metric.score:<10.3f} {safety_metric.score - orig_safety:+.3f}")

print("\nENHANCED SUMMARY:")
print(enhanced_summary)



Generating enhanced summary...


Output()

Output()

Output()

Output()


COMPARISON: ORIGINAL vs ENHANCED
Metric               Original   Enhanced   Change    
--------------------------------------------------
Summarization        0.353      0.294      -0.059
Coherence            0.662      0.661      -0.001
Tonality             0.829      0.848      +0.020
Safety               0.877      0.854      -0.023

ENHANCED SUMMARY:
In the grand tapestry of existence, the term 'noise' emerges as a most perplexing and multifarious concept, oscillating between the realms of delightful melody and maddening cacophony. The esteemed author, Alex Ross, embarks upon a profound exploration of this auditory phenomenon, tracing its etymological roots to notions of nuisance and nausea, whilst simultaneously acknowledging its capacity to inspire joy and reverence, as evidenced in sacred texts and artistic expressions. 

Through a historical lens, Ross reveals how various cultures articulate the essence of noise through their unique lexicons, thus underscoring the subjective n

In [14]:
# ============================================
# ENHANCEMENT - Self-correction with GPT-4o
# ============================================

print("Generating enhanced summary using GPT-4o...")

# Store original scores
orig_summarization = summarization_metric.score
orig_coherence = coherence_metric.score
orig_tonality = tonality_metric.score
orig_safety = safety_metric.score

# Create improved summary using GPT-4o (stronger model for refinement)
improve_prompt = f"""Please improve this summary based on the evaluation feedback below. Maintain Victorian English tone.

EVALUATION FEEDBACK:
- Summarization score was {orig_summarization:.3f}. Improve accuracy and coverage.
- Coherence score was {orig_coherence:.3f}. Improve flow and connections between ideas.

ORIGINAL SUMMARY:
{summary_result['Summary']}

Provide an improved version addressing the feedback. Keep Victorian English tone."""

completion = client.chat.completions.create(
    model="gpt-4o",  # Using stronger model for enhancement
    messages=[
        {"role": "developer", "content": "You improve summaries based on feedback."},
        {"role": "user", "content": improve_prompt}
    ],
    temperature=0.3
)

enhanced_summary = completion.choices[0].message.content

# Evaluate enhanced summary with same metrics
enhanced_test_case = LLMTestCase(
    input=document_text[:5000],
    actual_output=enhanced_summary
)

summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)

# Compare
print("\n" + "=" * 60)
print("COMPARISON: ORIGINAL vs ENHANCED (GPT-4o)")
print("=" * 60)
print(f"{'Metric':<20} {'Original':<10} {'Enhanced':<10} {'Change':<10}")
print("-" * 50)
print(f"{'Summarization':<20} {orig_summarization:<10.3f} {summarization_metric.score:<10.3f} {summarization_metric.score - orig_summarization:+.3f}")
print(f"{'Coherence':<20} {orig_coherence:<10.3f} {coherence_metric.score:<10.3f} {coherence_metric.score - orig_coherence:+.3f}")
print(f"{'Tonality':<20} {orig_tonality:<10.3f} {tonality_metric.score:<10.3f} {tonality_metric.score - orig_tonality:+.3f}")
print(f"{'Safety':<20} {orig_safety:<10.3f} {safety_metric.score:<10.3f} {safety_metric.score - orig_safety:+.3f}")

print("\nENHANCED SUMMARY (GPT-4o):")
print(enhanced_summary)


Generating enhanced summary using GPT-4o...


Output()

Output()

Output()

Output()


COMPARISON: ORIGINAL vs ENHANCED (GPT-4o)
Metric               Original   Enhanced   Change    
--------------------------------------------------
Summarization        0.235      0.188      -0.048
Coherence            0.762      0.742      -0.020
Tonality             0.832      0.813      -0.019
Safety               0.873      0.851      -0.022

ENHANCED SUMMARY (GPT-4o):
In the intricate tapestry of existence, the notion of 'noise' emerges as a most enigmatic and multifaceted concept, oscillating between the realms of harmonious melody and vexing cacophony. The esteemed author, Alex Ross, embarks upon a profound exploration of this auditory phenomenon, tracing its etymological roots to notions of nuisance and nausea, whilst simultaneously acknowledging its capacity to inspire joy and reverence, as evidenced in sacred texts and artistic expressions. Through a historical lens, Ross reveals how diverse cultures articulate the essence of noise through their unique lexicons, underscoring 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.

**Results**

Final Assignment Summary

Section	Result

Summary Generation is in Victorian English style

Summarization Score	0.294 → 0.235

Coherence Score	0.661 → 0.762 (+0.101)

Tonality Score	0.848 → 0.832

Safety Score	0.854 → 0.873 (+0.019)

Enhancement Model	GPT-4o (stronger model)



BUT THEN I RAN IT AGAIN AND THE SCORES WENT DOWN?

============================================================

COMPARISON: ORIGINAL vs ENHANCED (GPT-4o)

============================================================

Metric               Original   Enhanced   Change    

--------------------------------------------------

Summarization        0.235      0.188      -0.048

Coherence            0.762      0.742      -0.020

Tonality             0.832      0.813      -0.019

Safety               0.873      0.851      -0.022

**Going forward I would follow these strategies:**




1. **Fix sampling parameters**
Set temperature=0 to reduce randomness in the evaluator itself

2. **Use multiple evaluation runs**
Average scores across 3-5 runs instead of relying on a single score

3. **Provide examples in the prompt**
Few-shot examples help the judge be more consistent

4. **Use detailed rubrics**
Instead of vague criteria, specify exactly what each score level means

5. **Combine with exact evaluation**
Mix AI judges with functional tests (facts, format checks) that have deterministic results

6. **Human spot-checks** - Periodically verify AI judgments against human evaluation

TLDR: AI judges should complement, not replace, other evaluation methods.




# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
